## 课程实验

# D05：批量导入、失败与重试

Data Warehousing with Apache Doris · Level 1

[讲义](course.md) · [课程入口](../README.md)


## 首版执行范围

仅重建 d05_stream，使用本地已提交 CSV，不依赖 S3 凭据。执行前设置同一集群的 DW_BE_HTTP_URL。Kafka、CDC、对象存储和 Group Commit 实验仍在 integration-backlog.md 中明确列出，不由本 Lab 代替。


In [ ]:
from dw_course import WarehouseLab
from dw_course.runtime import COURSE_ROOT, fixture, expect, normalized
from dw_course.schema import ORDER_COLUMNS, order_ddl, order_rows

lab = WarehouseLab()
print("Course database:", lab.database)

from uuid import uuid4



## 1. 导入历史订单

显式 off_mode。先保留原始响应，再同时核对业务数据，不能只判断 HTTP 200。


In [ ]:
lab.execute("DROP TABLE IF EXISTS d05_stream")
ddl = order_ddl("d05_stream")
print(ddl)
lab.execute(ddl)
label = "dw_l1_" + uuid4().hex
columns = ",".join(ORDER_COLUMNS)
result = lab.stream_load("d05_stream", COURSE_ROOT / "datasets/orders.csv", label, columns)
print(result)
expect(result["Status"], "Success")
expect(result["NumberLoadedRows"], 10)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM d05_stream"), [(10,"1400.00")])


## 2. 同一批次用同 label 重试

在 label 仍有效的时间内立即重试。导入批次身份不等于永久业务去重，Unique Key 和事件版本在 D06 另行讨论。


In [ ]:
retry = lab.stream_load("d05_stream", COURSE_ROOT / "datasets/orders.csv", label, columns)
print(retry)
expect(retry["Status"], "Label Already Exists")
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM d05_stream"), [(10,"1400.00")])


## 3. 观察坏数据整批拒绝

新 label 导入两行，其中一行金额无法转换。strict_mode=true 且 max_filter_ratio=0；预期整批失败，原来的十行不变。


In [ ]:
rejected = lab.stream_load("d05_stream", COURSE_ROOT / "datasets/malformed_orders.csv",
                           "dw_bad_" + uuid4().hex, columns)
print(rejected)
expect(rejected["Status"], "Fail")
expect(rejected["NumberFilteredRows"], 1)
expect(lab.query("SELECT COUNT(*), SUM(order_amount) FROM d05_stream"), [(10,"1400.00")])
lab.close()


## 完成与排查

记录 ErrorURL 并在可信实验环境及时查看；不要将临时错误日志当永久拒收表。若出现 Publish Timeout 或不同状态，应保留响应核对事务与可见性，不能删除后重导掩盖问题。D09-A 将原始输入保留后再分流。
